# Housing Strand — Data Exploration & Training
**Team SCA-dream-soultion — Cold-Home Triage & Dataset Trust Auditor**

Run this notebook top-to-bottom. It explores the dataset, engineers features, makes an
**honest property-level split**, trains the three evaluated models, and writes every
deliverable (saved model, processed splits, filled JSONs, evidence dashboard, website bundle).

Then launch the website with `streamlit run app.py`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.data_pipeline import load_and_merge, build_feature_frame, make_property_split
df = load_and_merge(ROOT / 'data' / 'raw')
print('rows:', len(df), '| properties:', df['reference'].nunique())
df.head()

rows: 182750 | properties: 250


,reference,Sub-building,address,postcode,property_type,is_flat,year,month,day,avgTemperature,...,day_of_week,cold_risk,lag_temp,co2_missing,co2_imputed,pt_terraced,pt_semi_detached,pt_detached,co2_dropout_rate,high_dropout
0,109 belcroft rise,NaN,109 belcroft rise,ZZ6 4SX,detached,0,2023,1,1,16.78,...,6,1,NaN,0,623.310000,0,0,1,0.337893,0
1,109 belcroft rise,NaN,109 belcroft rise,ZZ6 4SX,detached,0,2023,1,2,17.81,...,0,1,16.78,0,651.520000,0,0,1,0.337893,0
2,109 belcroft rise,NaN,109 belcroft rise,ZZ6 4SX,detached,0,2023,1,3,17.32,...,1,1,17.81,1,625.110363,0,0,1,0.337893,0
3,109 belcroft rise,NaN,109 belcroft rise,ZZ6 4SX,detached,0,2023,1,4,17.22,...,2,1,17.32,0,517.190000,0,0,1,0.337893,0
4,109 belcroft rise,NaN,109 belcroft rise,ZZ6 4SX,detached,0,2023,1,5,17.34,...,3,1,17.22,0,677.280000,0,0,1,0.337893,0


## 1. Data-quality snapshot
Confirm the headline numbers and the CO₂ MNAR pattern.

In [2]:
print('cold-risk rate:', round(df['cold_risk'].mean(), 3))
for c in ['avgTemperature','avgHumidity','avgCo2','smart_meter_kwh','noise_db','survey_score']:
    print(f'{c:18s} missing {df[c].isna().mean():.1%}')
print('\nproperty types:'); print(df.groupby('reference')['property_type'].first().value_counts())
dropout = df.groupby('reference')['co2_missing'].mean()
print('\nhigh-dropout (>=50%) properties:', int((dropout >= 0.5).sum()))

cold-risk rate: 0.295
avgTemperature     missing 3.1%
avgHumidity        missing 5.0%
avgCo2             missing 38.4%
smart_meter_kwh    missing 2.5%
noise_db           missing 8.0%
survey_score       missing 97.2%

property types:
property_type
terraced         100
flat              70
semi-detached     55
detached          25
Name: count, dtype: int64

high-dropout (>=50%) properties: 40


## 2. Feature engineering & the honest split
We split by **property** (never by row) so `lag_temp` cannot leak across folds. We verify
that no property reference appears in both folds.

In [3]:
feat = build_feature_frame(df)
train_df, test_df, train_ids, test_ids = make_property_split(feat, seed=42)
overlap = set(train_df['reference']) & set(test_df['reference'])
print('train properties:', len(train_ids), '| test properties:', len(test_ids))
print('leakage check — overlapping properties:', len(overlap))

train properties: 187 | test properties: 63
leakage check — overlapping properties: 0


## 3. Train, evaluate & export everything
`build_submission.main()` trains the three models with 5-fold property-level CV, computes
the evidence views, fills both reference JSONs, and writes `reports/app_bundle.json` for the
website. (It also saves `saved_models/model_a_logistic_baseline.joblib` and the processed splits.)

In [4]:
from build_submission import main
ctx = main()

Loading + engineering features ...


  rows=182750 properties=250 cold_rate=0.295


Cross-validating the three models ...


  A=0.8412 B=0.8427 (CO2 lift +0.0015) C=0.9302 (lag leakage +0.089, row-split 0.9316)
  flat AUROC 0.6784 vs terraced 0.8295
Building upgrade ranking + sensor health + fairness ...


Writing reference JSONs + dashboard + app bundle ...


Done.


## 4. Headline findings

In [5]:
print('Model A (deployed, sensor-robust) AUROC :', ctx['m_a']['auroc'])
print('Model B (+CO2)  AUROC :', ctx['m_b']['auroc'], '(CO2 lift', f"{ctx['co2_lift']:+})")
print('Model C (+lag_temp) AUROC :', ctx['m_c']['auroc'], '(leakage', f"{ctx['lag_inflation']:+})")
print('flat AUROC vs terraced :', ctx['auroc_a']['flat'], 'vs', ctx['auroc_a']['terraced'])
print('overall verdict :', ctx['audit'].overall)
print('\nNext: run  ->  streamlit run app.py')

Model A (deployed, sensor-robust) AUROC : 0.8412
Model B (+CO2)  AUROC : 0.8427 (CO2 lift +0.0015)
Model C (+lag_temp) AUROC : 0.9302 (leakage +0.089)
flat AUROC vs terraced : 0.6784 vs 0.8295
overall verdict : CONDITIONAL

Next: run  ->  streamlit run app.py
